# CRNS Raw Counts Correction

**Yuxi Zhang** · Oct 18, 2026 · [@yuxi-research](https://github.com/yuxi-research)

📘 This notebook standardise raw cosmic-ray neutron counts, removing imfluence of atmospheric pressure, water vapor, incoming neutron flux, and location-related atmospeheric and cut-off rigidity differences, and combine the dataframe with ordinary kriged soil samples. The process align with the COSMOS guide VXXX (December 2025). Multiple alternative correction methods are provided.


⚙️ **Main Processing Steps (Counts Correction)**  
1. Neutron counts correction.
   * Atmospheric correction. Standardise to sea level.
   * Water vapor. Standardise to 0 absolute humidity.
   * Incoming neutron flux, standarised to JUNG with a reference value of 168, accounting for location-related cut-off rigidity differences among stations.

2. Combine the neutron counts data with kriged soil moisture and attributes extracted at rover locations.

📥 **Input**  
- `./data_raw/`  
   - `cutoff_rigidity.csv` # Cutoff rigidity of each survey. Manually downloaded from NMDB. Precise to day.

- `./data_lv2/{farm_name}/data/`  
   - `{farm_name}_{date}_rover_dataframe.csv` # Processed rover data, combined with neutron monitoring station (JUNG) data.
   - `{farm_name}_soilsamples_weighted.csv`# Vertically weighted soil data.


📤 **Output** 
- `./data_lv2/{farm_name}/data/`  
   - `{farm_name}_{date}_counts_corr.csv` # Corrected rover counts.



Spatial scaling: elevation, latitude, cutoff rigidity at different locations  
Temporal correction: pressure, vapor pressure, incoming flux intensity  

In [1]:
%load_ext autoreload
%autoreload 2

import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import geopandas as gpd
# from shapely.geometry import Point
import rasterio
import re
import datetime as dt
import seaborn as sns
import rioxarray as rxr

display=False


In [2]:
# Path settings
from global_settings import *
farm_name = os.environ.get("FARM_NAME", "Amaroo")
dirs = ProjectPaths(farm_name=farm_name)
proj_crs = dirs.proj_crs
inset_loc = dirs.inset_loc
buffer = BUFFER_DIS

In [3]:
dir_krig_ok = dirs.lv2_ok

## Read data


### Read boundary

In [4]:
# Read boundary
bound = gpd.read_file(os.path.join(dirs.lv1_shp, f'{farm_name}_boundary_epsg{proj_crs}.shp'))
bound = bound.to_crs(proj_crs)

### Read processed rover data
#### Search all rover file names and select a campagin

In [5]:
# Find all rover data and dates
regex = re.compile(rf'^{farm_name}_(.*)_rover_dataframe.csv$')
filesnames_rover = [f for f in os.listdir(dirs.lv2_data) if regex.match(f)]
filesnames_rover.sort()
print(filesnames_rover)
dates_str = [regex.match(f).group(1) for f in filesnames_rover]


#### Pipeline parameters

In [6]:
# Parameters
try:
    date_str
except NameError:
    date_str = dates_str[0]  # Index of the rover file to process


#### Select a single date for processing

In [7]:
filename = [f for f in filesnames_rover if date_str in f][0]
date = dt.datetime.strptime(date_str, '%Y-%m-%d').date()
farm_name, date_str, _, _ = re.split( '_', filename)
print(f'Processing file: {filename} for date: {date_str}')


#### Read the rover data file for the selected date

In [8]:
# Read rover data
df_rover = pd.read_csv(os.path.join(dirs.lv2_data, filename))
df_rover['geometry'] = gpd.points_from_xy(df_rover['LongDec'], df_rover['LatDec'], crs= '4326').to_crs(proj_crs)
df_rover['Date Time(UTC)'] = pd.to_datetime(df_rover['Date Time(UTC)'], format='%Y-%m-%d %H:%M:%S')

date_start = df_rover['Date Time(UTC)'].min()
date_end = df_rover['Date Time(UTC)'].max()

# Convert to GeoDataFrame
gdf_rover = gpd.GeoDataFrame(df_rover, geometry='geometry')


### Read cutoff rigidity data

In [9]:
# Read all cutoff rigidity from file and extract the value for the current date and farm
df_rig = pd.read_csv(dirs.raw_file_rig, parse_dates=['Date'])
rig = df_rig[(df_rig['Date'] == date_str) & (df_rig['Farm'] == farm_name)]['Rc'].values[0]
print(f'Cutoff rigidity for {farm_name} on {date_str}: {rig} GV')


## Neutron Counts Correction

### Prepare dataframe

In [10]:
# Remove duplicates and sort by index
gdf_rover = gdf_rover[~gdf_rover.duplicated(subset=['LatDec', 'LongDec'])]
gdf_rover = gdf_rover.sort_index(ascending=True)

# Raw counts per second (cts/s) for each channel, then sum across channels to get total counts per second
gdf_rover['CountsRaw'] = sum([(gdf_rover[f'N{i}Cts'] * 60) / gdf_rover[f'N{i}ETsec'] for i in range(1, 17)])

# Rename columns for consistency
# PTB110_mb: Pressure in millibars (mb). 1mb = 1hPa.
# T_CS215: Temperature in degrees Celsius (°C)
# RH_CS215: Relative Humidity in percentage (%)
gdf_rover.rename(columns={'PTB110_mb': 'Pressure', 'T_CS215': 'Temperature', 'RH_CS215': 'Humidity'}, inplace=True)

### Explotory Data Analysis

#### Check data distribution

In [11]:
# Check the distribution of pressure, temperature, relative humidity, reference neutron flux, and raw counts
gdf_rover[['Pressure', 'Temperature', 'Humidity', 'NMCounts', 'CountsRaw']].describe().round(2)

#### Check pressure change with time and altitude

In [12]:
# Checking only
### -- Check Reference Pressure at rover altitude and the actual pressure measured by rover -- ###
# Barometric formula to calculate reference pressure at rover altitude. gdf_rover['Alt'] - Altitude in meters
gdf_check = gdf_rover[['Date Time(UTC)', 'Alt', 'Pressure']].copy()

gdf_check['AltRefPressure'] = 1013.25 * np.exp((-9.80665 * 0.0289644 * gdf_check['Alt']) / (8.31447 * 288.15))


# Difference
gdf_check['DeltaPressure'] = gdf_check['Pressure'] - gdf_check['AltRefPressure']

fig, axes = plt.subplots(figsize=(8, 6), nrows=2, ncols=1)
sns.lineplot(ax=axes[0], data=gdf_check, x='Date Time(UTC)', y='Pressure', label='Actual Pressure', color='blue')
sns.lineplot(ax=axes[0], data=gdf_check, x='Date Time(UTC)', y='AltRefPressure', label='Altitude-Reference Pressure', color='black')

sns.lineplot(ax=axes[1], data=gdf_check, x='Date Time(UTC)', y='DeltaPressure', label='Pressure Difference (Actual - Reference)', color='red')
plt.legend(bbox_to_anchor=(1.05, 1))
plt.title('Distribution of Reference Pressure and Measured Pressure')
plt.tight_layout()

### **Pressure and elevation correction**
* **Method 1 (COSMOS Guide 2015)**: Zreda 2012 equation and Tirado-Bueno 2021 for beta calculation. 
  * Standardise to sea level. This corrects pressure change caused by temporal change and elevation difference at the same time.
  * The calculated factor is used as a multiplier.


In [13]:
# Method: Tirado-Bueno 2021
def calc_beta(rig):
    return 0.0074 - 0.0001 * np.log(rig)

ref_pressure = 1013.25 # Reference pressure in hPa
beta = calc_beta(rig)

factor_method1_pressure_elevation = np.exp(beta * (gdf_rover['Pressure'] - ref_pressure))

# Select this method for pressure correction. This factor is applied as N_corr = N_raw * FactorPressure
gdf_rover['FactorPressure'] = factor_method1_pressure_elevation


* **Method 2 (Desilet 2006)**: Based on Desilet 2006 elevation scaling but is also applicable for pressure correction, because the variation in neutron counts from both source are a result of atmospheric depth.
    * Attenuation length is the inverse of integral of attenuation coefficient (beta) between two locations.
    * The attenuation coefficient is calculated using the curve of high-energy neutron attenuation coefficients in Desilets 2006 (equation 5-7).



In [14]:
#### ---- Pressure and elevation correction: Correct pressure to sea level directly. ---- ####
# This function calculates the integral of attenuation coefficient. The denominator for each of the atmospheric depth in equation 4 in Desilets 2006.
def calc_integral_beta(x, rig_local):
    # Coefficients for equation 4， Table 2 in Desilets 2006
    n = 1.0177e-02
    alpha = 1.0207e-01
    k = -3.9527e-01
    a0 = 8.5236e-06
    a1 = -6.3670e-07
    a2 = -7.0814e-09
    a3 = -9.9182e-09
    a4 = 9.9250e-10
    a5 = 2.4925e-11
    a6 = 3.8615e-12
    a7 = -4.8194e-13
    a8 = -1.5371e-14

    term1 = n * (1 + np.exp(-alpha * rig_local**(-k)))**(-1) * x
    term2 = 1/2 * (a0 + a1*rig_local + a2*rig_local**2) * x**2
    term3 = 1/3 * (a3 + a4*rig_local + a5*rig_local**2) * x**3
    term4 = 1/4 * (a6 + a7*rig_local + a8*rig_local**2) * x**4
    return term1 + term2 + term3 + term4

# Zreda 2012 and Hawdon 2014 used high-energy neutron attenuation coefficients in Desilets 2006 (equation 5-7). The same equation is used here.
def calc_factor_elev(atmos_depth, rig, x_ref=1033):
    # x_ref =1033: This is the atmospheric depth at sea level, which is the reference condition for elevation scaling.

    # Note: Their is an error in Desilet 2006 Eq 7, so I calculated the attenuation length from Eq 4 and substitude it into Eq 1 to calculate the elevation factor.
    # atten_length =(x_ref - atmos_depth) / (calc_denominator(x2, rig) - calc_denominator(x1, rig)) (# Eq 4)
    # fac_elev = np.exp((x_ref - atmos_depth)/atten_length) (#Eq 1)
    # Combining the two equations above, we can directly calculate the elevation factor as follows:
    scale_elev = np.exp((calc_integral_beta(x_ref, rig) - calc_integral_beta(atmos_depth, rig)))
    return scale_elev

In [15]:
# Change to True to apply alternative method for pressure and elevation correction. 
# This method uses the integral of attenuation coefficient to calculate the scaling factor for elevation correction.

# Calculate atmospheric depth. A simple unit conversion is applied to convert pressure to atmospheric depth in g/cm^2.
gdf_rover['AtmosDepth'] = gdf_rover['Pressure'] / (9.80655 / 10)

# Calculate elevation scaling factor using the function defined above. This will be used to correct the neutron counts for the effect of elevation.
factor_method2_pressure_elevation_orig = calc_factor_elev(gdf_rover['AtmosDepth'], rig)

#Note the above factor is applied as N_corr = N_raw / ScalePressureElevation
# Here we apply the elevation scaling factor to the raw counts to get the corrected counts for elevation. Uncomment the line below to apply the correction.
factor_method2_pressure_elevation = 1 / factor_method2_pressure_elevation_orig


* **Method 3**: Correct measured pressure to the reference pressure at measurement altitude (Zreda 2012), then scale the altitude to the sea level (Desilet 2006).
Approximation of method 2, to demonstrate that altitude correction and pressure correction are theoretically the same, and no need to apply seperately.

**Pressure conrrection**
Temporal pressure difference at the same location.
The corrections were performed following the procedures documented by Zreda et al., (2012).
The correction for atmospheric pressure variation is;
$$ f_{p} = exp[{\beta (p - p^{ref})}]$$
```python
press_corr = np.exp(beta_ref * (cos['Pressure'] - pressure_ref))
```


In [16]:
# Calculate attenuation coefficient at the target atmospheric depth.

# Method: Desilet 2003, 2006.
def calc_beta2(atmos_depth, rig):
    '''
    Calculate attenuation coefficient at the target atmospheric depth.
    Function name 'calc_beta2' is used to avoid confusion with the previous 'calc_beta' function.
    The calculation follows Eq 8 in Desilet 2003, but using modified parameters in Desilets 2006 (Table 2 for spallation reactions) 
    '''
    # Coefficients Table 2 in Desilets 2006 for spallation reactions.
    n = 1.0177e-02
    alpha = 1.0207e-01
    k = -3.9527e-01
    a0 = 8.5236e-06
    a1 = -6.3670e-07
    a2 = -7.0814e-09
    a3 = -9.9182e-09
    a4 = 9.9250e-10
    a5 = 2.4925e-11
    a6 = 3.8615e-12
    a7 = -4.8194e-13
    a8 = -1.5371e-14

    # Eq 8 in Desilet 2003.
    term1 = n * (1 + np.exp(-alpha * rig**(-k)))**(-1) 
    term2 =  (a0 + a1*rig + a2*rig**2) * atmos_depth
    term3 =  (a3 + a4*rig + a5*rig**2) * atmos_depth**2
    term4 = (a6 + a7*rig + a8*rig**2) * atmos_depth**3

    return term1 + term2 + term3 + term4


In [17]:
# Calculate atmospheric depth. A simple unit conversion is applied to convert pressure to atmospheric depth in g/cm^2.
gdf_rover['AtmosDepth'] = gdf_rover['Pressure'] / (9.80655 / 10)

# Calculate reference pressure at rover altitude using the barometric formula. This will be used to calculate the reference atmospheric depth.
gdf_rover['AltRefPressure'] = 1013.25 * np.exp((-9.80665 * 0.0289644 * gdf_rover['Alt']) / (8.31447 * 288.15))

# Convert reference pressure to reference atmospheric depth
gdf_rover['RefAtmosDepth'] = gdf_rover['AltRefPressure'] / (9.80655 / 10)

# Method 3: Using effective attenuation length between actual and reference atmospheric depths
factor_method3_pressure = gdf_rover.apply(lambda x: 1/calc_factor_elev(x['AtmosDepth'], rig, x_ref=x['RefAtmosDepth']), axis=1)
#Note this factor is applied by N_corr = N_raw * Scale2Pressure

# Method 3: Using beta at reference atmospheric depth
beta2 = calc_beta2(gdf_rover['RefAtmosDepth'], rig)
factor_method3_pressure = np.exp(beta2 * (gdf_rover['Pressure'] - gdf_rover['AltRefPressure']))
#Note this factor is applied by N_corr = N_raw * factor_method3_pressure

# Calculate elevation scaling factor at refrence atmospheric depth
factor_method3_elevation = calc_factor_elev(gdf_rover['RefAtmosDepth'], rig)
#Note this factor is applied as N_corr = N_raw / factor_method3_elevation

# The combined scaling factor for pressure and elevation correction. Used as multiplier.
factor_method3_pressure_elevation = factor_method3_pressure / factor_method3_elevation

#### Uncomment to select correction method

In [18]:
#### Uncomment to select this method for pressure correction. This factor is applied as N_corr = N_raw * FactorPressureElevation
#### gdf_rover['FactorPressure'] = factor_method1_pressure_elevation
#### gdf_rover['FactorPressure'] = factor_method2_pressure_elevation
#### gdf_rover['FactorPressure'] = factor_method3_pressure_elevation

#### Compare methods 1 and 2 for pressure and elevation correction.

In [19]:
# Check the pressure scale factor difference between the two calculation methods
(factor_method1_pressure_elevation - factor_method2_pressure_elevation).abs().mean()

In [20]:
plt.figure(figsize=(4, 3))
plt.plot(gdf_rover['Date Time(UTC)'], factor_method1_pressure_elevation, label='Method 1', color='blue')
plt.plot(gdf_rover['Date Time(UTC)'], factor_method2_pressure_elevation, label='Method 2', color='green')
plt.plot(gdf_rover['Date Time(UTC)'], factor_method3_pressure_elevation, label='Method 3', color='red')
plt.legend()

### **Vapor correction**
Due to the vapor pressure difference between the measurement day and the reference day.  
* **Method 1: COSMOS Guide.**

In [21]:
### Atmospheric water vapor correction, relative to reference conditions (0 g/m^3) ###
vp_ref = 0
# Saturation vapor pressure in hPa at temperature Temperature.
gdf_rover['VPSat'] = 6.112 * np.exp((17.27 * gdf_rover['Temperature']) / (gdf_rover['Temperature'] + 237.3))
# Actual vapor pressure in hPa or mb. Humidity: Relative humidity in percentage.
gdf_rover['VP'] =gdf_rover['VPSat'] * (gdf_rover['Humidity'] / 100)
# Absolute humidity in g/m^3. The formula is derived from the ideal gas law, where the constant 216.74 is used to convert the units appropriately.
gdf_rover['AbsHumidity'] = 216.74 * gdf_rover['VP'] / (gdf_rover['Temperature'] + 273.15)
# Scale factor for vapor pressure correction, relative to reference conditions (0 g/m^3)
gdf_rover['FactorVap'] = 1 + 0.0054 * (gdf_rover['AbsHumidity'] - vp_ref)
#Note this factor is applied by N_corr = N_raw / ScaleVap

* **Method 2: Rosolem 2013**. Formulas are the same. Negligible differences exist in the actual vapor pressure formula parameters.


$$
VP = \frac{2165}{T + 273.15}  \times \left[ 0.6108 \times \exp\left( \frac{17.27\, T}{T + 237.3} \right) \right] \times  \frac{RH}{100}
$$

$$ f_{vp} = 1 + 0.0054 * (VP - VP^{ref})
$$
Rosolem et al. (2013)

* VP:Actual vapor pressure（Pa）
* T: Air temperature（°C）
* RH: Relative Humidity（%）
* 2165: Unit conversion factor（hPa × 1000 × ... → Pa）
* [*]: Saturated vapor pressure at the given temperature
* VP_ref: Vapor pressure difference between the measurement day and calibration day. 
* Reference time: Any reference time vp e.g., the time of calirbation.

### **Intensity correction**
Method 1: COSMOS GUide

In [22]:
#### Incoming neutron flux correction, relative to reference conditions (ref_flux) ###
ref_flux = 168 # Reference flux value

# Reference neutron monitoring (MN) station cutoff rigidity at JUNG. Change this value if using a different reference station.
rig_nm = 4.5 # Source: https://www.nmdb.eu/station/jung/
lat_nm = 46.55 # Latitude in decimal degrees (N)
alt_nm = 3570 # m

In [23]:
def calc_x(lat, alt):
    """
    Calculate the atmospheric depth (x) at a given latitude and altitude.
    """
    theta = lat * np.pi / 180
    c1 = 9.780327
    c2 = 0.0518592
    c3 = 0.0000567
    c4 = -3.086e-6
    c5 = 4.194e-10
    c6 = 10.1325
    c7 = -0.00002256
    c8 = 0.5359502
    rho_rock = 2670

    g_loc = c1 + c2 * np.sin(theta)**2 - c3 * np.sin(2 * theta)**2 + (c4 + c5 * rho_rock) * gdf_rover['Alt'] 
    P_loc = c6 * (1 + c7 * gdf_rover['Alt'])** (c8 * g_loc)
    x_loc = P_loc / g_loc
    
    return x_loc


In [24]:
def calc_tau(x_loc, x_nm, rig, rig_nm):
    '''
    Calculate the scaling factor accounting for the difference in atmospheric depth and cutoff rigidity between the rover location and the reference neutron monitoring station.
    Reference: COSMOS Guide.
    Parameters:
        x_loc: Atmospheric depth at the rover location (g/cm^2)
        x_nm: Atmospheric depth at the reference neutron monitoring station (g/cm^2)
        rig: Cutoff rigidity at the rover location (GV)
        rig_nm: Cutoff rigidity at the reference neutron monitoring station (GV)
    Returns:
        tau: Scaling factor
    '''
    # Parameters
    c1 = 0.0005085
    c2 = 0.0064
    c3 = 1.8855
    c4 = 1.3e-5
    c5 = 1.2237
    epsilon = 1

    # Equation
    tau = epsilon * (1-c1*x_loc)/(1-c1*x_nm) * (1-np.exp(-(c2* x_loc + c3)* rig **(c4 * x_loc - c5))) / (1-np.exp(-(c2* x_nm + c3)* rig_nm **(c4 * x_nm - c5)))

    return tau

In [25]:
x_loc = calc_x(gdf_rover['LatDec'], gdf_rover['Alt'])
x_nm = calc_x(lat_nm, alt_nm)  # Atmospheric depth at sea level for the same latitude
tau = calc_tau(x_loc, x_nm, rig, rig_nm)

# Calculate the scaling factor for incoming neutron flux correction, relative to reference conditions (ref_flux)
gdf_rover['FactorInten'] = 1 / (tau * (gdf_rover['NMCounts'] / ref_flux - 1) + 1)

* **Method 2: Hawdon 2014**
Temporal intensity difference.
(Howdon 2014, Eq 3-5)

The correction for incoming neutron flux intensity is;
$$ f_{i} = \frac{l_{m}}{l_{ref}} $$

where lm is the selected neutron monitor counting rate at any particular point in time and lref is a
reference counting rate for the same neutron monitor from an arbitrary fixed point in time which is the
date of the survey. The Neutron monitor data is sourced from the Neutron Monitor Database (NMDB;
www.nmdb.eu) of Lomnicky Stit Observatory in Slovakia.

$$
Rc_{corr} = -0.075 (Rc - Rc_{ref}) + 1
$$

$$
fi_{Rc} = (fi - 1) Rc_{corr} + 1
$$


In [26]:
# Old method
#### Incoming nutron flux correction, relative to reference conditions (ref_intensity) ###
factor_inten_unscaled = gdf_rover['NMCounts'] / ref_flux
# Calculate corrected cutoff rigidity factor, relative to reference conditions (rig_mn)
rig_corr = -0.075 * (rig - rig_nm) + 1
# Calculate correction factor for neutron flux correction. The original factor is used as diviser.
factor_inten_orig =  (factor_inten_unscaled-1) * rig_corr + 1

# The final correction is inverse of the original factor, and applied as a multiplier to the raw counts.
factor_inten = 1/factor_inten_orig

In [27]:
plt.figure(figsize=(12, 3))
plt.plot(gdf_rover['Date Time(UTC)'], gdf_rover['FactorInten'], label='Method 1: COSMOS Guide', color='blue', lw=1)
plt.plot(gdf_rover['Date Time(UTC)'], factor_inten, label='Method 2: Hawdon 2014', color='red', ls='--', lw=1)
plt.xlabel('Date Time (UTC)')
plt.ylabel('Factor Intensity')
plt.title('Neutron Flux Correction Factors')
plt.legend()
plt.show()

### **Vegetation correction**
Vegetation correction is not applied as the survey was conducted in pre-season and the vegetaion impact is negligible. Further versions would account for vegetation correction.

### **Final Scaling Factors**

In [28]:
gdf_rover['FactorTotal'] = gdf_rover['FactorPressure'] * gdf_rover['FactorVap'] * gdf_rover['FactorInten']
gdf_rover['CountsCorr'] = gdf_rover['CountsRaw'] * gdf_rover['FactorTotal']

### **Export corrected neutron counts**

In [29]:
# Export
gdf_rover.to_csv(os.path.join(dirs.lv2_data, f'{farm_name}_{date_str}_counts_corr.csv'), index=False)

## Analysis of neutron counts

In [30]:
col_factor = [f for f in gdf_rover.columns if f.startswith('Factor')]
gdf_rover[col_factor].describe().round(2)

In [31]:
# Line plot of the scale factors over time
plt.figure(figsize=(12, 6))
for col in col_factor:
    plt.plot(gdf_rover['Date Time(UTC)'], gdf_rover[col], label=col)
plt.xlabel('Date Time (UTC)')
plt.ylabel('Scale Factor')
plt.title('Scale Factors Over Time')
plt.legend()

In [32]:
tmp = gdf_rover[['CountsRaw', 'CountsCorr']]

# Normalize each column in tmp for better visualization
tmp = tmp.transform(lambda x: (x - x.min()) / (x.quantile(0.95) - x.quantile(0.05)))

tmp.hist(bins=50, figsize=(8, 3), alpha=0.7)

In [33]:
sns.set(style="whitegrid")
plt.figure(figsize=(4, 4))
sns.scatterplot(data=gdf_rover, x='CountsRaw', y='CountsCorr', s=10, alpha=0.8, hue='SurveyDate')
ax = plt.gca()

# sns.scatterplot(ax=ax, data=gdf_rover, x='CountsRaw', y='CountsCorr_NDVI', s=50, alpha=0.5, c='orange', label='NCorr with NDVI')
ax.plot([100,600], [100, 600], color='grey', linestyle='--', linewidth=1)  # Add a reference line
ax.set_ylim(ax.get_xlim())
ax.set_title(f"Counts Raw vs Counts Corr ({farm_name} {date_str})")
ax.set_xlabel('Counts Raw')
ax.set_ylabel('Counts Corr')
ax.set_aspect('equal', adjustable='box')